# LangChain: Q&A over Documents (Modernized with LCEL)

An example might be a tool that would allow you to query a product catalog for items of interest.

> **Note**: This notebook uses modern LCEL RAG chains instead of the deprecated `RetrievalQA`, `VectorstoreIndexCreator`, and `langchain.llms.OpenAI`.

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [2]:
# Set the model variable
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [3]:
import os
from pathlib import Path
data_dir =Path(os.getcwd()).parent /"data"

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from IPython.display import display, Markdown

In [7]:
file = data_dir / 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')

In [8]:
# Load documents and create vector store (replaces VectorstoreIndexCreator)
embeddings = OpenAIEmbeddings()
docs = loader.load()
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [9]:
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

Using a modern LCEL RAG chain instead of the deprecated `VectorstoreIndexCreator.query()` and `gpt-3.5-turbo-instruct`.

In [10]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the following context:\n\n"
    "{context}\n\n"
    "Question: {question}"
)

# LCEL RAG chain (replaces VectorstoreIndexCreator + RetrievalQA)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke(query)

In [11]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                | Summary                                                                                                                                                                                                 |
|-------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Men's Tropical Plaid Short-Sleeve Shirt | A lightweight, wrinkle-resistant shirt made of 100% polyester, offering UPF 50+ sun protection. Features include front and back cape venting and two front bellows pockets. Imported design.             |
| Men's Plaid Tropic Shirt, Short-Sleeve  | Designed for fishing, this shirt is made of 52% polyester and 48% nylon, providing UPF 50+ sun protection. It is wrinkle-free, quick-drying, and machine washable. Includes cape venting and bellows pockets. |
| Men's TropicVibe Shirt, Short-Sleeve    | A relaxed fit shirt with a shell of 71% nylon and 29% polyester, and a polyester knit mesh lining. Offers UPF 50+ sun protection, is wrinkle-resistant, and features cape venting and bellows pockets.      |
| Sun Shield Shirt                       | Slightly fitted shirt made of 78% nylon and 22% Lycra Xtra Life fiber, providing UPF 50+ sun protection. It wicks moisture, is abrasion-resistant, and fits over swimsuits. Recommended by The Skin Cancer Foundation. |

## Step By Step

In [12]:
# Reload documents for step-by-step walkthrough
loader = CSVLoader(file_path=file,  encoding='utf-8')

In [13]:
docs = loader.load()

In [14]:
docs[0]

Document(metadata={'source': 'e:\\GIT_ROOT\\Learning\\DLAI-shortcourse_notebooks\\courses\\C11 - LangChain for LLM Application Development\\data\\OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [15]:
# Using langchain_openai instead of deprecated langchain.embeddings
embeddings = OpenAIEmbeddings()

In [16]:
embed = embeddings.embed_query("Hi my name is Harrison")

In [17]:
print(len(embed))

1536


In [18]:
print(embed[:5])

[-0.021954253315925598, 0.006774455308914185, -0.018215758726000786, -0.03919148072600365, -0.014013086445629597]


In [19]:
# Using InMemoryVectorStore instead of deprecated DocArrayInMemorySearch
db = InMemoryVectorStore.from_documents(
    docs,
    embeddings
)

In [20]:
query = "Please suggest a shirt with sunblocking"

In [21]:
docs = db.similarity_search(query)

In [22]:
len(docs)

4

In [23]:
docs[0]

Document(id='7308c899-79c8-412b-9bcb-d852b56861e9', metadata={'source': 'e:\\GIT_ROOT\\Learning\\DLAI-shortcourse_notebooks\\courses\\C11 - LangChain for LLM Application Development\\data\\OutdoorClothingCatalog_1000.csv', 'row': 255}, page_content=': 255\nname: Sun Shield Shirt by\ndescription: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. \n\nSize & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.\n\nFabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.\n\nAdditional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.\n\nSun Protection That Won\'t Wear Off\nOur high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun\'s harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective U

In [24]:
retriever = db.as_retriever()

In [25]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [26]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))])


In [27]:
# Using llm.invoke() instead of deprecated llm.call_as_llm()
response = llm.invoke(f"{qdocs} Question: Please list all your \
shirts with sun protection in a table in markdown and summarize each one.")
response = response.content

In [28]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                               | Description                                                                                                           | Size & Fit                  | Fabric & Care                                                                 | Additional Features                                                                 |
|------------------------------------|-----------------------------------------------------------------------------------------------------------------------|-----------------------------|-------------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| Sun Shield Shirt                   | High-performance sun shirt with UPF 50+ protection, blocks 98% of harmful UV rays.                                    | Slightly Fitted: Softly shapes the body. Falls at hip. | 78% nylon, 22% Lycra Xtra Life fiber. Handwash, line dry.                           | Moisture-wicking, abrasion-resistant, fits over swimsuit, imported.                  |
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable shirt with UPF 50+ protection, originally designed for fishing, great for travel.                     | Not specified               | 52% polyester, 48% nylon. Machine washable and dryable.                           | Wrinkle-free, quick-drying, front and back cape venting, two front bellows pockets.  |
| Men's TropicVibe Shirt, Short-Sleeve | Lightweight sun-protection shirt with UPF 50+, ideal for hot weather and strong UV rays.                               | Traditional Fit: Relaxed through the chest, sleeve, and waist. | Shell: 71% Nylon, 29% Polyester. Lining: 100% Polyester knit mesh. Machine wash and dry. | Wrinkle-resistant, front and back cape venting, two front bellows pockets, imported. |
| Men's Tropical Plaid Short-Sleeve Shirt | Lightest hot-weather shirt with UPF 50+ protection, traditional fit, and wrinkle-resistant fabric.                      | Traditional Fit: Relaxed through the chest, sleeve, and waist. | 100% polyester. Machine wash and dry.                                              | Front and back cape venting, two front bellows pockets, imported.                    |

Each shirt provides UPF 50+ sun protection, blocking 98% of the sun's harmful rays, ensuring you stay protected while enjoying outdoor activities.

In [29]:
# LCEL RAG chain with retriever (replaces RetrievalQA.from_chain_type)
retriever = db.as_retriever()

qa_prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the following context:\n\n"
    "{context}\n\n"
    "Question: {question}"
)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [30]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [31]:
# invoke() replaces deprecated .run()
response = qa_chain.invoke(query)

In [32]:
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                | Description                                                                                                           |
|-------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| Men's Tropical Plaid Short-Sleeve Shirt | A lightweight, wrinkle-resistant shirt made of 100% polyester, offering UPF 50+ sun protection with front and back cape venting and two front bellows pockets. |
| Men's Plaid Tropic Shirt, Short-Sleeve  | A comfortable shirt made of 52% polyester and 48% nylon, designed for fishing and travel, with UPF 50+ protection, wrinkle-free fabric, and quick-drying properties. |
| Men's TropicVibe Shirt, Short-Sleeve    | A relaxed fit shirt with a shell of 71% nylon and 29% polyester, featuring UPF 50+ protection, wrinkle resistance, and front and back cape venting. |
| Sun Shield Shirt                       | A slightly fitted shirt made of 78% nylon and 22% Lycra Xtra Life fiber, offering UPF 50+ protection, moisture-wicking, and abrasion resistance. |

In [33]:
# Same query using the rag_chain defined earlier
response = rag_chain.invoke(query)
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                | Description                                                                                                           |
|-------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| Men's Tropical Plaid Short-Sleeve Shirt | A lightweight, wrinkle-resistant shirt made of 100% polyester, offering UPF 50+ sun protection with front and back cape venting and two front bellows pockets. |
| Men's Plaid Tropic Shirt, Short-Sleeve  | A comfortable shirt originally designed for fishing, made of 52% polyester and 48% nylon, featuring UPF 50+ sun protection, wrinkle-free fabric, and quick-drying properties. |
| Men's TropicVibe Shirt, Short-Sleeve    | A sun-protection shirt with a traditional fit, made of 71% nylon and 29% polyester, featuring UPF 50+ sun protection, wrinkle resistance, and front and back cape venting. |
| Sun Shield Shirt                       | A slightly fitted shirt made of 78% nylon and 22% Lycra Xtra Life fiber, offering UPF 50+ sun protection, moisture-wicking, and abrasion resistance, recommended by The Skin Cancer Foundation. |

In [34]:
# Recreate vector store with explicit embedding (equivalent to the old VectorstoreIndexCreator with embedding param)
loader = CSVLoader(file_path=file, encoding='utf-8')
docs = loader.load()
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# Rebuild chain with the new retriever
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)